# E1: BCE в полосе границы

Старт из EMA `disentangle_b2_li760_r8_long/ckpt/best.pt`; новый optimizer/scheduler.
3 × 24 000 примеров; LR encoder 1e-5, JPEG/head 3e-5; 25% негативов.
Все эпохи full-frame; JPEG 30%, Q60–99, без сдвига сетки. Sampling обычный.

E1 добавляет `0.2 * boundary_bce`: BCE финальной маски в квадратной полосе ±4 px
на сетке RGB 760 (dilation − erosion, окно 9×9; GT > 0.5 только для выбора полосы).
BCE использует исходный target; нормировка на площадь полосы отдельно по изображению,
затем среднее по всему батчу. Пустая полоса даёт 0, основной loss негативов сохраняется.
Coarse patch/edge-loss не меняются. E0 отличается только `boundary_weight=0`.

Выбор best сохраняет исходную weighted AIC (вес малых масок 1.6).
Сравнивать также обычные AIC/Dice/FPR при исходном пороге 0.3984375, cls=0, area=0,
и фиксированные по исходной модели группы Dice. Holdout/test не использовать для подбора.

Настройки устройства/batch/accumulation берутся из `.env`; держите их одинаковыми в E0/E1.
Последняя ячейка запускает полный дотюн на сервере; повторный запуск продолжает свой last.pt.


In [ ]:
import numpy as np  # Import before torch on Windows (MKL initialization).
import os
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if not (ROOT / 'src').is_dir():
    raise RuntimeError('Open this notebook from the project root or notebooks directory')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

os.chdir(ROOT)

from src.config import load_experiment_config

experiment = 'experiments/disentangle_b2_li760_r8_boundary_ft'
cfg = load_experiment_config(ROOT / 'configs' / f'{experiment}.yaml')
print('Run:', cfg.run_name)
print('Encoder / RGB:', cfg.model.encoder, cfg.dataset.image_size)
print('Epochs / full passes:', cfg.train.epochs, cfg.train.full_pass_epochs)
print('Devices / batch per GPU / accumulation:', cfg.train.devices, cfg.train.batch_size, cfg.train.grad_accum_steps)
print('Data:', cfg.paths.data_path)
print('Runs:', cfg.paths.runs_path)
cfg


In [ ]:
import torch
from src.training.builders import build_model
from src.budget import count_gflops
from src.eval.protocol import EvaluationProtocol

protocol = EvaluationProtocol.load(cfg.dataset.protocol_path)
print('Train/development:', len(protocol.rows('train')), len(protocol.rows('development')))
native_size = (1080, 1920)
with torch.device('meta'):
    budget_model = build_model(cfg.model, aux_weight=cfg.loss.aux_weight, pretrained=False).eval()
    gflops = count_gflops(budget_model, cfg.dataset.image_size,
                          native_size=native_size)
del budget_model
assert gflops <= 100, f'{gflops:.2f} GFLOPs exceeds 100'
print(f'Full inference: {gflops:.3f} GFLOPs')
if native_size is not None:
    print('Reference native JPEG size:', native_size, '; larger sources may exceed 100 GFLOPs')


In [ ]:
checkpoint = cfg.paths.runs_path / cfg.train.finetune_from
last = cfg.paths.runs_path / cfg.run_name / 'ckpt' / 'last.pt'
if cfg.train.resume and last.is_file():
    print('Resume:', last)
else:
    assert checkpoint.is_file(), f'Missing source checkpoint: {checkpoint}'
    print('Initialize:', checkpoint, cfg.train.finetune_weights)
print('Boundary weight / radius:', cfg.loss.boundary_weight, cfg.loss.boundary_radius)
print('Existing patch / edge weights:', cfg.loss.patch_weight, cfg.loss.edge_weight)
print('LR encoder / JPEG / head:', cfg.train.encoder_lr, cfg.train.jpeg_lr, cfg.train.head_lr)
print('Loss settings:', cfg.loss)


In [ ]:
from src.training.engine import run_experiment

run = run_experiment(cfg)
run.summary
